In [1]:
# Load the autoreload extension
%load_ext autoreload
# Set it to automatically reload all modules every time you run a cell
%autoreload 2

In [2]:
import torch
import tiktoken
import sys
sys.path.append('../scr')
from dataLoader import GPTDataset
from attentionLayers import MultiHeadMaskedAttention
from torch.utils.data import Dataset, DataLoader
from gptmodel import LayerNorm, TransformerBlock, GPTModel
torch.manual_seed(42)

In [3]:
print(torch.__version__)

2.5.1


In [4]:
torch.cuda.is_available()

True

# Check tokenizer

In [5]:
tokenizer = tiktoken.get_encoding("gpt2")

In [6]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 5372, 13]


In [7]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace.


In [8]:
with open("../training_data/the-verdict.txt", "r") as f:
    raw_text = f.read()

In [9]:
dataset = GPTDataset(txt = raw_text, tokenizer=tokenizer, max_length=100, stride=10)

In [10]:
data_iter = iter(dataset)
first_batch = next(data_iter)

In [11]:
first_batch

(tensor([   40,   367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,
           257,  7026, 15632,   438,  2016,   257,   922,  5891,  1576,   438,
           568,   340,   373,   645,  1049,  5975,   284,   502,   284,  3285,
           326,    11,   287,   262,  6001,   286,   465, 13476,    11,   339,
           550,  5710,   465, 12036,    11,  6405,   257,  5527, 27075,    11,
           290,  4920,  2241,   287,   257,  4489,    64,   319,   262, 34686,
         41976,    13,   357, 10915,   314,  2138,  1807,   340,   561,   423,
           587, 10598,   393, 28537,  2014,   198,   198,     1,   464,  6001,
           286,   465, 13476,     1,   438,  5562,   373,   644,   262,  1466,
          1444,   340,    13,   314,   460,  3285,  9074,    13, 46606,   536]),
 tensor([  367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,   257,
          7026, 15632,   438,  2016,   257,   922,  5891,  1576,   438,   568,
           340,   373,   645,  1049,  5975,   284,

# Check DataLoader

In [12]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True) # drop_last = True drops the last batch if it is shorter than the specified batch_size

In [13]:
dataloader_iter = iter(dataloader)

In [14]:
next(dataloader_iter)

[tensor([[ 8759,  2763,    26,  ...,   286,  2376, 26927],
         [ 2123, 10813,  1701,  ...,  1169,   691,  2134],
         [ 5365, 31655,    26,  ...,   616, 35957,    25],
         ...,
         [  438, 18108,   407,  ...,   683,   736,   284],
         [  517,    13,   383,  ...,     0,   198,   198],
         [  290,  1807,   683,  ...,  1762,    30,  2011]]),
 tensor([[ 2763,    26,   393,  ...,  2376, 26927,   616],
         [10813,  1701,   198,  ...,   691,  2134,  7163],
         [31655,    26,   475,  ..., 35957,    25,   366],
         ...,
         [18108,   407, 11196,  ...,   736,   284,   262],
         [   13,   383,  3200,  ...,   198,   198,     1],
         [ 1807,   683, 32081,  ...,    30,  2011, 29483]])]

# Check Multihead masked attention layer

In [15]:
d_in = 5
d_out = 4
context_length = 10
dropout = 0.2
num_heads = 2
qkv_bias = False
masked_attention = MultiHeadMaskedAttention(d_in = d_in, d_out = d_out, \
                                            context_length = context_length, \
                                            dropout = dropout, num_heads = num_heads, \
                                            qkv_bias = qkv_bias)

In [16]:
x = torch.rand(10, 5)

In [17]:
batch = torch.stack((x, x), dim = 0)

In [18]:
batch.shape

torch.Size([2, 10, 5])

In [19]:
outputs = masked_attention.forward(batch)

In [20]:
print(outputs.shape)

torch.Size([2, 10, 4])


# Check layer norm

In [21]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch)
mean = out_ln.mean(dim = -1, keepdim=True)
var = out_ln.var(dim = -1, keepdim=True)
print(mean)
print(var)

tensor([[[-2.3842e-08],
         [ 8.3447e-08],
         [ 8.9407e-09],
         [ 0.0000e+00],
         [-5.8115e-08],
         [-1.7881e-08],
         [ 7.1526e-08],
         [ 2.3842e-07],
         [-1.1921e-08],
         [-9.5367e-08]],

        [[-2.3842e-08],
         [ 8.3447e-08],
         [ 8.9407e-09],
         [ 0.0000e+00],
         [-5.8115e-08],
         [-1.7881e-08],
         [ 7.1526e-08],
         [ 2.3842e-07],
         [-1.1921e-08],
         [-9.5367e-08]]], grad_fn=<MeanBackward1>)
tensor([[[0.9999],
         [0.9997],
         [0.9999],
         [0.9998],
         [0.9997],
         [0.9999],
         [0.9997],
         [0.9996],
         [0.9999],
         [0.9999]],

        [[0.9999],
         [0.9997],
         [0.9999],
         [0.9998],
         [0.9997],
         [0.9999],
         [0.9997],
         [0.9996],
         [0.9999],
         [0.9999]]], grad_fn=<VarBackward0>)


In [22]:
out_ln = ln(batch)

In [23]:
out_ln

tensor([[[ 1.0714, -0.2688,  1.0160, -1.1255, -0.6932],
         [-0.0069, -0.2595, -1.5199,  0.7800,  1.0064],
         [-0.8007, -0.5051,  1.7352, -0.2015, -0.2279],
         [-1.1050, -0.2937,  1.3957,  0.6115, -0.6085],
         [ 1.6690, -0.2634, -0.7155,  0.0988, -0.7888],
         [-0.8025,  1.3081,  0.2624,  0.4163, -1.1842],
         [-0.5851,  0.2463,  1.5019, -1.1578, -0.0053],
         [-0.8375,  1.1325, -0.0954,  0.8965, -1.0961],
         [-0.5625, -0.4076, -0.0447, -0.7174,  1.7322],
         [-0.5203,  1.0006,  0.2123, -1.4515,  0.7588]],

        [[ 1.0714, -0.2688,  1.0160, -1.1255, -0.6932],
         [-0.0069, -0.2595, -1.5199,  0.7800,  1.0064],
         [-0.8007, -0.5051,  1.7352, -0.2015, -0.2279],
         [-1.1050, -0.2937,  1.3957,  0.6115, -0.6085],
         [ 1.6690, -0.2634, -0.7155,  0.0988, -0.7888],
         [-0.8025,  1.3081,  0.2624,  0.4163, -1.1842],
         [-0.5851,  0.2463,  1.5019, -1.1578, -0.0053],
         [-0.8375,  1.1325, -0.0954,  0.8965, 

# check TransformerBlock

In [24]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [25]:
x = torch.rand(2, 4, 768)

In [26]:
block = TransformerBlock(GPT_CONFIG_124M)

In [27]:
output = block(x)

In [28]:
x.shape

torch.Size([2, 4, 768])

In [29]:
output.shape

torch.Size([2, 4, 768])

# Check GPTMODEL

In [30]:
model = GPTModel(GPT_CONFIG_124M)

In [31]:
batch = torch.tensor([[6109, 3626, 6100, 345], [6109, 1110, 6622, 2576]])

In [32]:
batch.shape

torch.Size([2, 4])

In [33]:
output = model(batch)

In [34]:
output.shape

torch.Size([2, 4, 50257])

# Check the number of parameters

In [48]:
total_params = sum([p.numel() for p in model.parameters()]) # check number of elements

In [49]:
total_params

163009536

In [64]:
num_atten_params = sum([p.numel() for block in model.trf_blocks for p in block.att.parameters()])

In [65]:
num_ffn_params = sum([p.numel() for block in model.trf_blocks for p in block.ff.parameters()])

In [66]:
print(num_atten_params)
print(num_ffn_params)

28320768
56669184


# Generate text

In [73]:
test = [1,2,3]

In [100]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        
        logits = logits[:, -1, :] # get the logits for the last token
        probas = torch.softmax(logits, dim = -1)
        idx_next = torch.argmax(probas, dim = -1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim = 1)
    return idx

In [101]:
start_context = 'Hello, I am'
encoded = tokenizer.encode(start_context)
print(encoded)

[15496, 11, 314, 716]


In [102]:
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

In [103]:
encoded_tensor

tensor([[15496,    11,   314,   716]])

In [105]:
generated_encoded = generate_text_simple(model=model,
                     idx = encoded_tensor,
                     max_new_tokens = 10,
                     context_size = 5)

In [111]:
tokenizer.decode(generated_encoded.squeeze().tolist())

'Hello, I am gum IndianapolisAgainstEXTFalse Bundesliga Brain deceased Antarcticlords'